# regime 2 / QLoRA (backprop) — 4-bit frozen ViT-Base + LoRA on CIFAR-10
Same as LoRA but the frozen base is **4-bit (nf4)** via bitsandbytes (GPU-only). Adam updates the FP16 adapters through the dequantized base. Saves `results/qlora.json`.

## Step 0 — Setup (installs transformers / peft / bitsandbytes)

In [ ]:
import os, time, math, json, subprocess, sys
def _pip(*pkgs): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
_pip('transformers', 'peft', 'accelerate', 'bitsandbytes')   # bitsandbytes only needed by qLoRA (GPU)
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: regime 2 needs a GPU (HF ViT-Base; qLoRA needs CUDA bitsandbytes). Use SMOKE=True for a tiny check.')

## Config

In [ ]:
METHOD, QUANTIZE = 'qlora', True
LR = 1e-3
METHOD_CFG = dict(LR=LR)


# =============================== CONFIG (shared across the 4 methods) ===============================
# Frozen base + LoRA adapters are IDENTICAL across methods (same trainable set) -> fair race.
MODEL_NAME   = 'google/vit-base-patch16-224'    # 86M, ImageNet; classifier head replaced with 10-way
NUM_CLASSES, IMG_SIZE = 10, 224                 # CIFAR-10, upsampled to 224
LORA_R, LORA_ALPHA, LORA_DROPOUT = 4, 8, 0.0
LORA_TARGETS       = ['query', 'value']         # attention projections (paper: separable-ish; sweep to 'dense' for FFN)
LORA_LAST_K_LAYERS = 2                           # adapt only the last K of 12 blocks -> small P_trainable, cheaper probes

# Fair match = EQUAL WALL-CLOCK. Run the 4 notebooks on separate A100 runtimes, then compare.
TIME_BUDGET_HOURS = 5.0
BATCH             = 64
SEED              = 0
EVAL_BATCH        = 256
LOG_EVERY_SEC, CKPT_EVERY_SEC, PLOT_EVERY_SEC = 30, 120, 3600   # log / checkpoint / live-plot cadence
USE_DRIVE, DRIVE_SUBDIR = True, 'Section8_regime2'


SMOKE = False
if SMOKE:                                        # ~30 s tiny check (no Drive, tiny data, 1 LoRA layer)
    LORA_LAST_K_LAYERS = 1
    TIME_BUDGET_HOURS = 30/3600
    LOG_EVERY_SEC, CKPT_EVERY_SEC, PLOT_EVERY_SEC = 3, 5, 999
    EVAL_BATCH, USE_DRIVE, N_SMOKE_IMAGES = 128, False, 512
    if 'M_PROBES' in dir(): M_PROBES = 16
# ===================================================================================================

## Step 1 — Storage (Drive) + paths

In [ ]:
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed -> local /content (will NOT survive disconnect):', e)
        STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR, CKPT_DIR, FIG_DIR = (os.path.join(STORE, s) for s in ('results', 'checkpoints', 'figures'))
for d in (RESULTS_DIR, CKPT_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
RESULTS_PATH = os.path.join(RESULTS_DIR, f'{METHOD}.json')
CKPT_PATH    = os.path.join(CKPT_DIR,    f'{METHOD}.pt')
print('storing under:', STORE)

## Step 2 — Data (CIFAR-10 → 224)

In [ ]:
import torchvision
_tr = torchvision.datasets.CIFAR10('./data', train=True,  download=True)
_te = torchvision.datasets.CIFAR10('./data', train=False, download=True)
Xtr = torch.tensor(_tr.data).permute(0, 3, 1, 2).float().div(255.).to(device)   # (50000,3,32,32)
Ytr = torch.tensor(_tr.targets).to(device)
Xte = torch.tensor(_te.data).permute(0, 3, 1, 2).float().div(255.).to(device)
Yte = torch.tensor(_te.targets).to(device)
if SMOKE:
    Xtr, Ytr, Xte, Yte = Xtr[:N_SMOKE_IMAGES], Ytr[:N_SMOKE_IMAGES], Xte[:N_SMOKE_IMAGES], Yte[:N_SMOKE_IMAGES]

def _prep(x):                                    # (B,3,32,32) -> (B,3,224,224), ViT normalisation (mean/std 0.5)
    x = F.interpolate(x, size=IMG_SIZE, mode='bilinear', align_corners=False)
    return (x - 0.5) / 0.5

def fresh_batch(n):
    idx = torch.randint(0, Xtr.shape[0], (n,), device=device)
    return _prep(Xtr[idx]), Ytr[idx]
print('CIFAR-10 train', tuple(Xtr.shape), '| test', tuple(Xte.shape))

## Step 3 — Frozen ViT-Base + LoRA adapters

In [ ]:
from transformers import ViTForImageClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
torch.manual_seed(SEED)
if QUANTIZE:                                     # qLoRA: 4-bit frozen base (needs CUDA + bitsandbytes)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    _base = ViTForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True, quantization_config=bnb)
    from peft import prepare_model_for_kbit_training
    _base = prepare_model_for_kbit_training(_base)
else:
    _base = ViTForImageClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True).to(device)

_n = _base.config.num_hidden_layers
_layers = list(range(_n - LORA_LAST_K_LAYERS, _n))
_lcfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                   target_modules=LORA_TARGETS, layers_to_transform=_layers,
                   modules_to_save=['classifier'], bias='none')
model = get_peft_model(_base, _lcfg)
model.eval()                                     # deterministic forwards (LoRA_DROPOUT=0 anyway)
def trainable_params(): return {n: p for n, p in model.named_parameters() if p.requires_grad}
TP = trainable_params()
P    = sum(p.numel() for p in model.parameters())
Ptr  = sum(v.numel() for v in TP.values())
def forward_fn(x): return model(pixel_values=x).logits
model.print_trainable_parameters()
print(f'total P = {P:,} | trainable (LoRA + head) Ptr = {Ptr:,} | LoRA on last {LORA_LAST_K_LAYERS} layers, targets {LORA_TARGETS}')

## Step 4 — Budget + provenance

In [ ]:
Mstar = math.ceil((Ptr + 1) / 3)
M = (Mstar if M_PROBES == 'mstar' else int(M_PROBES)) if METHOD == 'three_factor' else 0
if METHOD == 'three_factor':
    print(f'three-factor: Ptr={Ptr:,} -> M*={Mstar:,}; using M={M:,} probes/step (each probe = full ViT-Base forward).')
CFG = dict(METHOD=METHOD, QUANTIZE=QUANTIZE, MODEL_NAME=MODEL_NAME, SEED=SEED, BATCH=BATCH,
           LORA_R=LORA_R, LORA_ALPHA=LORA_ALPHA, LORA_TARGETS=LORA_TARGETS,
           LORA_LAST_K_LAYERS=LORA_LAST_K_LAYERS, TIME_BUDGET_HOURS=TIME_BUDGET_HOURS)
CFG.update(METHOD_CFG)
if METHOD == 'three_factor': CFG['M'] = M

## Step 6 — Trainer

In [ ]:
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
def train_step():
    xb, yb = fresh_batch(BATCH)
    opt.zero_grad(); loss = F.cross_entropy(forward_fn(xb), yb); loss.backward(); opt.step()
    return loss.item()
def ckpt_state():      return {'tp': {k: v.detach().cpu() for k, v in TP.items()}, 'opt': opt.state_dict()}
def load_ckpt_state(d):
    with torch.no_grad():
        for k in TP: TP[k].data.copy_(d['tp'][k].to(device))
    opt.load_state_dict(d['opt'])

## Step 7 — Eval / checkpoint / live-plot

In [ ]:
@torch.no_grad()
def eval_metrics():
    correct = tot = 0; loss_sum = 0.0
    for i in range(0, Xte.shape[0], EVAL_BATCH):
        xb = _prep(Xte[i:i+EVAL_BATCH]); yb = Yte[i:i+EVAL_BATCH]
        logits = forward_fn(xb)
        correct += (logits.argmax(-1) == yb).sum().item(); tot += yb.numel()
        loss_sum += F.cross_entropy(logits, yb, reduction='sum').item()
    return loss_sum/tot, correct/tot

In [ ]:
def _atomic_save(o, p): t = p + '.tmp'; torch.save(o, t); os.replace(t, p)
def save_checkpoint(step, elapsed, logs):
    _atomic_save({'loop': {'step': step, 'elapsed_sec': elapsed, 'logs': logs},
                  'method': ckpt_state(), 'config': CFG}, CKPT_PATH)
def maybe_resume():
    if os.path.exists(CKPT_PATH):
        d = torch.load(CKPT_PATH, map_location=device); load_ckpt_state(d['method']); L = d['loop']
        print(f'[resume] {METHOD}: step {L["step"]:,}, elapsed {L["elapsed_sec"]/3600:.2f}h -> continuing.')
        return L['step'], L['elapsed_sec'], L['logs']
    return 0, 0.0, []

In [ ]:
def live_plot(logs, final=False):
    if not logs: return
    hrs = [l[0]/3600 for l in logs]; tr=[l[3] for l in logs]; te=[l[4] for l in logs]; ac=[l[5] for l in logs]
    if not final: clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hrs, tr, '-', color='#999', label='train loss'); ax[0].plot(hrs, te, 'o-', color='#C62828', label='test loss')
    ax[0].set_xlabel('elapsed hours'); ax[0].set_ylabel('loss'); ax[0].set_title(f'{METHOD}: loss vs time'); ax[0].legend()
    ax[1].plot(hrs, ac, 'o-', color='#1F3864'); ax[1].set_xlabel('elapsed hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('accuracy vs time')
    plt.tight_layout()
    try: fig.savefig(os.path.join(FIG_DIR, f'{METHOD}_progress.png'), dpi=90)
    except Exception as e: print('savefig failed:', e)
    plt.show()
    last = logs[-1]; print(f'[h={last[0]/3600:.2f}] step={last[1]:,} test_loss={last[4]:.4f} acc={last[5]:.3f}')

## Step 8 — Train (equal wall-clock) + save results

In [ ]:
def write_results(step, elapsed, logs):
    te = [l[4] for l in logs]; ac = [l[5] for l in logs]
    summary = dict(initial_loss=logs[0][4], final_loss=logs[-1][4], best_loss=min(te),
                   reduction_pct=100*(logs[0][4]-min(te))/max(logs[0][4], 1e-9),
                   final_acc=logs[-1][5], best_acc=max(ac))
    peak = torch.cuda.max_memory_allocated()/1e6 if device == 'cuda' else float('nan')
    out = dict(meta=dict(method=METHOD, device=device, seed=SEED, P=P, P_trainable=Ptr, config=CFG,
                         note='all MEASURED-here', wall_clock_sec=elapsed, total_steps=step,
                         epochs=step*BATCH/50000, peak_mem_mb=peak),
               curve=dict(t_sec=[l[0] for l in logs], step=[l[1] for l in logs], epoch=[l[2] for l in logs],
                          train_loss=[l[3] for l in logs], test_loss=te, test_acc=ac),
               summary=summary)
    with open(RESULTS_PATH, 'w') as f: json.dump(out, f, indent=2)
    print('wrote', RESULTS_PATH)

def run():
    step, elapsed0, logs = maybe_resume()
    t0 = time.time(); budget = TIME_BUDGET_HOURS*3600
    last_ckpt = last_plot = last_log = elapsed0
    if not logs:
        l0, a0 = eval_metrics(); logs.append((elapsed0, step, 0.0, l0, l0, a0))
    tl = logs[-1][3]; printed = False
    while True:
        elapsed = elapsed0 + (time.time() - t0)
        if elapsed >= budget: break
        tl = train_step(); step += 1
        if not printed:
            dt = time.time() - t0
            print(f'[{METHOD}] step 1 = {dt:.2f}s -> ~{int(budget/max(dt,1e-9)):,} steps fit the budget.'); printed = True
        if elapsed - last_log  >= LOG_EVERY_SEC:  el, ac = eval_metrics(); logs.append((elapsed, step, step*BATCH/50000, tl, el, ac)); last_log = elapsed
        if elapsed - last_ckpt >= CKPT_EVERY_SEC: save_checkpoint(step, elapsed, logs); last_ckpt = elapsed
        if elapsed - last_plot >= PLOT_EVERY_SEC: live_plot(logs); last_plot = elapsed
    elapsed = elapsed0 + (time.time() - t0)
    el, ac = eval_metrics(); logs.append((elapsed, step, step*BATCH/50000, tl, el, ac))
    save_checkpoint(step, elapsed, logs); live_plot(logs, final=True); write_results(step, elapsed, logs)
    print(f'DONE {METHOD}: {step:,} steps, {elapsed/3600:.2f}h, final test acc {ac:.3f}, best loss {min(l[4] for l in logs):.4f}')

run()

## How to run
- **`SMOKE=True` first** (Config) for a ~30 s check, then `False`.
- **Runtime → Run all** on an A100. First cell pip-installs `transformers`/`peft`/`bitsandbytes`.
- Disconnect-safe: checkpoints the **adapters only** (small) to Drive every ~2 min and auto-resumes; hourly live loss-vs-time graph saved to Drive.
- Same frozen base + identical LoRA config across all four methods = a fair race. Compare with `05_compare_results.ipynb`.